In [2]:
import os
%pwd
os.chdir("../")
%pwd

'd:\\STUDY\\CDS\\End-to-End-ML-project-with-MLFlow'

In [3]:
%pwd

'd:\\STUDY\\CDS\\End-to-End-ML-project-with-MLFlow'

In [4]:
# Configure MLflow to log runs to the DagsHub tracking server using these credentials.
 
os.environ["MLFLOW_TRACKING_URI"]= "https://dagshub.com/irdgvaiiy/End-to-End-ML-project-with-MLFlow.mlflow"
os.environ["MLFLOW_TRACKING_USERNAME"] = "irdgvaiiy"
os.environ["MLFLOW_TRACKING_PASSWORD"] = "328ff6a58f22dea81cfb0e113b1fafe30a3ce9d4"

In [11]:
# 4. Update the entity
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class ModelEvaluationConfig:
    root_dir: Path
    test_data_path: Path
    model_path: Path
    all_params: dict
    metric_file_name: Path
    target_column: str
    mlflow_uri: str

In [12]:
# 5. Update the configuration manager in src config

from mlProject.constants import *
from mlProject.utils.common import read_yaml, create_directories, save_json

In [13]:
# 5. Update the configuration manager in src config

class ConfigurationManager:
    def __init__(
        self,
        config_filepath=CONFIG_FILE_PATH,
        paramas_filepath=PARAMS_FILE_PATH,
        schema_filepath=SCHEMA_FILE_PATH,
    ):

        self.config = read_yaml(config_filepath)
        self.params = read_yaml(paramas_filepath)
        self.schema = read_yaml(schema_filepath)

        create_directories([self.config.artifacts_root])

    
    def get_model_evaluation_config(self) -> ModelEvaluationConfig:
        config = self.config.model_evaluation
        params = self.params.ElasticNet
        schema = self.schema.TARGET_COLUMN

        create_directories([config.root_dir])

        self.get_model_evaluation_config = ModelEvaluationConfig(
            root_dir=config.root_dir,
            test_data_path=config.test_data_path,
            model_path=config.model_path,
            all_params=params,
            metric_file_name = config.metric_file_name,
            target_column=schema.name,
            mlflow_uri="https://dagshub.com/irdgvaiiy/End-to-End-ML-project-with-MLFlow.mlflow"
        )

        return self.get_model_evaluation_config

In [14]:
# 5. Update the configuration manager in src config

import os
import pandas as pd
import mlflow
import mlflow.sklearn
import numpy as np
import joblib
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from urllib.parse import urlparse

In [15]:
class ModelEvaluation:
    def __init__(self, config: ModelEvaluationConfig):
        self.config = config

    def eval_metrics(self, actual, pred):
        rmse = np.sqrt(mean_squared_error(actual, pred))
        mae = mean_absolute_error(actual, pred)
        r2 = r2_score(actual, pred)
        return rmse, mae, r2

    def log_into_mlflow(self):
        test_data = pd.read_csv(self.config.test_data_path)
        model = joblib.load(self.config.model_path)

        test_x = test_data.drop([self.config.target_column], axis=1)
        test_y = test_data[self.config.target_column]

        mlflow.set_registry_uri(self.config.mlflow_uri)
        tracking_url_type_store = urlparse(mlflow.get_tracking_uri()).scheme

        with mlflow.start_run():
            predicted_qualities = model.predict(test_x)
            (rmse, mae, r2) = self.eval_metrics(test_y, predicted_qualities)

            scores = {"rmse":rmse, "mae":mae, "r2":r2}
            save_json(path=Path(self.config.metric_file_name), data=scores)

            mlflow.log_params(self.config.all_params)

            mlflow.log_metric("rmse", rmse)
            mlflow.log_metric("r2", r2)
            mlflow.log_metric("mae", mae)


            # Model registry does not work with file store
            if tracking_url_type_store != "file":
                #Register the model
                # There are other ways to use the Model Registry, which depends on the use case,
                # Please refer to the doc for more information:
                # https://mlflow.org/docs/latest/model-registry.html#api-workflow
                mlflow.sklearn.log_model(model, "model", registered_model_name="ElasticnetModel")
            
            else:
                mlflow.sklearn.log_model(model, "model")



In [16]:
try:
    config = ConfigurationManager()
    model_evaluation_config = config.get_model_evaluation_config()
    model_evaluation_config = ModelEvaluation(config = model_evaluation_config)
    model_evaluation_config.log_into_mlflow()
except Exception as e:
    raise e

[2026-05-25 18:43:27,186: INFO: common: yaml file: config\config.yaml Loaded successfully]
[2026-05-25 18:43:27,190: INFO: common: yaml file: params.yaml Loaded successfully]
[2026-05-25 18:43:27,198: INFO: common: yaml file: schema.yaml Loaded successfully]
[2026-05-25 18:43:27,202: INFO: common: created directory at: artifacts]
[2026-05-25 18:43:27,204: INFO: common: created directory at: artifacts/model_evaluation]
[2026-05-25 18:43:28,853: INFO: common: json file saved at: artifacts\model_evaluation\metrics.json]


2026/05/25 18:43:30 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/25 18:43:35 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
Registered model 'ElasticnetModel' already exists. Creating a new version of this model...
2026/05/25 18:43:56 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: ElasticnetModel, version 2
Created version '2' of model 'ElasticnetModel'.


🏃 View run honorable-sheep-292 at: https://dagshub.com/irdgvaiiy/End-to-End-ML-project-with-MLFlow.mlflow/#/experiments/0/runs/5d12a8eda412459388a607baadb4a302
🧪 View experiment at: https://dagshub.com/irdgvaiiy/End-to-End-ML-project-with-MLFlow.mlflow/#/experiments/0
